In [1]:
import json
import random

from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, TextGenerationPipeline


[2025-04-09 12:23:22,679] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


In [2]:
lima = load_dataset("GAIR/lima", split="train", trust_remote_code=True)

In [3]:
def is_single_turn(example):
    return isinstance(example["conversations"], list) and len(example["conversations"]) == 2

In [4]:
lima_single_turn = lima.filter(is_single_turn)
print(f"Single-turn examples: {len(lima_single_turn)}")

Single-turn examples: 1000


In [5]:
sampled = random.sample(list(lima_single_turn), 150)
responses = [ex["conversations"][1] for ex in sampled]

In [6]:
model_path = "../saves/backward_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path)
pipeline = TextGenerationPipeline(model=model, tokenizer=tokenizer, device=2)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:2


In [7]:
def build_prompt(response):
    return f"### Response:\n{response}\n\n### Instruction:"

In [8]:
prompts = [build_prompt(resp) for resp in responses]

batch_size = 30
augmented_pairs = []

for i in tqdm(range(0, len(prompts), batch_size), desc="Generating instructions", total=len(prompts) // batch_size):
    batch_prompts = prompts[i : i + batch_size]

    batch_outputs = pipeline(batch_prompts, max_new_tokens=100, do_sample=True, temperature=0.7)

    for output, response in zip(batch_outputs, responses[i : i + batch_size]):
        generated = output[0]["generated_text"]

        if "### Instruction:" in generated:
            instruction = generated.split("### Instruction:")[-1].strip()
        else:
            instruction = generated.strip()

        augmented_pairs.append((instruction, response))

Generating instructions:   0%|          | 0/5 [00:00<?, ?it/s]

In [9]:
print("\n===== 5 Sampled Instruction-Response Pairs =====\n")
for i in range(5):
    print(f"Example {i+1}:")
    print("Instruction:", augmented_pairs[i][0])
    print("Response:", augmented_pairs[i][1])
    print("-" * 60)


===== 5 Sampled Instruction-Response Pairs =====

Example 1:
Instruction: For Win32 programs, Windows implements pre-emptive multitasking. Its implementation is based upon the ```message loop``` architecture of every Windows program.

The duty of every program is to endlessly run in a loop in which a call to the ```GetMessage``` function is performed. This function call looks whether a message to this process is in the queue. If there is one, it is retrieved (```GetMessage```), optionally translated (```TranslateMessage
Response: For Win16 programs, Windows implemented co-operative multitasking. Its implementation was based upon the &quot;message loop&quot; architecture of every Windows program.

The duty of every program was to endlessly run in a loop in which a call to the ```GetMessage``` function was performed. This function call looks whether a message to this process is in the queue. If there is one, it is retrieved (```GetMessage```), optionally translated (```TranslateMessage`

In [11]:
save_path = "self_augmented_lima_150.json"

with open(save_path, "w", encoding="utf-8") as f:
    for instruction, response in augmented_pairs:
        item = {
            "input": response.strip(),   # 输入是回答（response）
            "output": instruction.strip()  # 输出是生成的指令
        }
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n✅ 已保存到 {save_path}，共 {len(augmented_pairs)} 条数据。")


✅ 已保存到 self_augmented_lima_150.json，共 150 条数据。


: 